# Global MMLU preprocessing

This notebook loads Global MMLU for Russian, Chinese, Italian, Spanish, and Hindi; explores cultural-sensitivity labels and subject categories; cleans and deduplicates MCQ records; maps `cultural_sensitivity_label` into our `regionality` labels; applies fixed-seed subject-category sampling; and saves SFT-compatible JSONL files under `multilingual/data/processed/global_mmlu/`.

# Imports

In [4]:
from pathlib import Path
import json
import numpy as np
import os
import random

import pandas as pd
! pip install matplotlib
import matplotlib.pyplot as plt
from datasets import load_dataset

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.7/118.7 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 3.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 362.6/362.6 kB 3.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 3.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 3.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.8/122.8 kB 4.0 MB/s eta 0:00:00


In [5]:
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

In [6]:
REPO_ROOT = Path('/scratch/workspace/emma/standard-project-m2-mystery-machine')
OUTPUT_DIR = REPO_ROOT / 'multilingual/data/processed/global_mmlu'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATASET_NAME = 'CohereLabs/Global-MMLU'
LANGUAGE_CONFIGS = {
    'russian': 'ru',
    'chinese': 'zh',
    'italian': 'it',
    'spanish': 'es',
    'hindi': 'hi',
}
SOURCE_SPLIT = 'test'
RANDOM_SEED = 42
VALID_ANSWERS = {'A', 'B', 'C', 'D'}
VALID_REGIONALITIES = {'regional', 'regional_implicit', 'region_agnostic'}
REGIONALITY_MAP = {'CS': 'regional', 'CC': 'regional_implicit'}

# Adjust after inspecting the exact subject_category names below.
SUBJECT_CATEGORY_WEIGHTS = {
    'STEM': 0.3,
    'Natural Science': 0.3,
    'Natural Sciences': 0.3,
    'Humanities': 1.0,
    'Social Sciences': 1.0,
    'Business & Economics': 1.0,
    'Business and Economics': 1.0,
    'Law': 1.0,
    'Medical & Health': 1.0,
    'Medicine & Health': 1.0,
    'Health': 1.0,
    'Other': 0.5,
}
DEFAULT_CATEGORY_WEIGHT = 0.5
HF_TOKEN = os.environ.get('HF_TOKEN')

In [7]:
frames = []

for language, config in LANGUAGE_CONFIGS.items():
    print(f'Loading {DATASET_NAME} | config={config} | split={SOURCE_SPLIT}')
    ds = load_dataset(DATASET_NAME, config, split=SOURCE_SPLIT, token=HF_TOKEN)
    df_lang = ds.to_pandas()
    df_lang['language'] = language
    df_lang['source_split'] = SOURCE_SPLIT
    df_lang['source_idx'] = range(len(df_lang))
    frames.append(df_lang)

df_raw = pd.concat(frames, ignore_index=True)
print(f'Loaded rows: {len(df_raw)}')
display(df_raw.head())

Loading CohereLabs/Global-MMLU | config=ru | split=test
Loading CohereLabs/Global-MMLU | config=zh | split=test
Loading CohereLabs/Global-MMLU | config=it | split=test
Loading CohereLabs/Global-MMLU | config=es | split=test
Loading CohereLabs/Global-MMLU | config=hi | split=test
Loaded rows: 70210


,sample_id,subject,subject_category,question,option_a,option_b,option_c,option_d,answer,required_knowledge,time_sensitive,reference,culture,region,country,cultural_sensitivity_label,is_annotated,language,source_split,source_idx
0,abstract_algebra/test/0,abstract_algebra,STEM,Найдите степень для заданного расширения поля ...,0,4,2,6,B,[],[],[],[],[],[],-,False,russian,test,0
1,abstract_algebra/test/1,abstract_algebra,STEM,"Пусть p = (1, 2, 5, 4)(2, 3) в S_5. Найдите ин...",8,2,24,120,C,[],[],[],[],[],[],-,False,russian,test,1
2,abstract_algebra/test/2,abstract_algebra,STEM,Найдите все нули в указанном конечном поле зад...,0,1,"0,1","0,4",D,[],[],[],[],[],[],-,False,russian,test,2
3,abstract_algebra/test/3,abstract_algebra,STEM,Заявление 1 | Фактор-группа неабелевой группы ...,"Правда, Правда","Ложь, Ложь","Правда, Ложь","Ложь, Правда",B,"['none', 'none', 'none', 'none', 'none']","['No', 'No', 'No', 'No', 'No']","['-', '-', '-', '-', '-']",[],[],[],CA,True,russian,test,3
4,abstract_algebra/test/4,abstract_algebra,STEM,Найдите произведение данных многочленов в данн...,2x^2 + 5,6x^2 + 4x + 6,0,x^2 + 1,B,[],[],[],[],[],[],-,False,russian,test,4


# Explore

In [8]:
print("Columns:")
print(df_raw.columns.tolist())

print("\nRaw shape:")
print(df_raw.shape)

print("\ncultural_sensitivity_label values:")
display(df_raw["cultural_sensitivity_label"].value_counts(dropna=False).to_frame("n_examples"))

print("\nsubject_category values:")
display(df_raw["subject_category"].value_counts(dropna=False).sort_index().to_frame("n_examples"))

print("\nlanguage counts:")
display(df_raw["language"].value_counts().sort_index().to_frame("n_examples"))

print("\nsubject counts, top 30:")
display(df_raw["subject"].value_counts(dropna=False).head(30).to_frame("n_examples"))

Columns:
['sample_id', 'subject', 'subject_category', 'question', 'option_a', 'option_b', 'option_c', 'option_d', 'answer', 'required_knowledge', 'time_sensitive', 'reference', 'culture', 'region', 'country', 'cultural_sensitivity_label', 'is_annotated', 'language', 'source_split', 'source_idx']

Raw shape:
(70210, 20)

cultural_sensitivity_label values:


,n_examples
cultural_sensitivity_label,
-,55960
CA,10290
CS,3960



subject_category values:


,n_examples
subject_category,
Business,3595
Humanities,23525
Medical,7525
Other,4415
STEM,15765
Social Sciences,15385



language counts:


,n_examples
language,
chinese,14042
hindi,14042
italian,14042
russian,14042
spanish,14042



subject counts, top 30:


,n_examples
subject,
professional_law,7670
moral_scenarios,4475
miscellaneous,3915
professional_psychology,3060
high_school_psychology,2725
high_school_macroeconomics,1950
elementary_mathematics,1890
moral_disputes,1730
prehistory,1620


In [9]:
print("Cultural sensitivity labels:")
display(df_raw["cultural_sensitivity_label"].value_counts(dropna=False).to_frame("n_examples"))

print("\nSubject categories:")
display(df_raw["subject_category"].value_counts(dropna=False).sort_index().to_frame("n_examples"))

print("\nSubject x cultural label, top rows:")
display(
    df_raw.groupby(["subject", "cultural_sensitivity_label"])
    .size()
    .reset_index(name="n_examples")
    .sort_values("n_examples", ascending=False)
    .head(80)
)

Cultural sensitivity labels:


,n_examples
cultural_sensitivity_label,
-,55960
CA,10290
CS,3960



Subject categories:


,n_examples
subject_category,
Business,3595
Humanities,23525
Medical,7525
Other,4415
STEM,15765
Social Sciences,15385



Subject x cultural label, top rows:


,subject,cultural_sensitivity_label,n_examples
131,professional_law,-,7420
117,moral_scenarios,-,4225
111,miscellaneous,-,3665
137,professional_psychology,-,2810
74,high_school_psychology,-,2475
62,high_school_macroeconomics,-,1700
37,elementary_mathematics,-,1640
114,moral_disputes,-,1480
125,prehistory,-,1370
122,philosophy,-,1305


In [10]:
REGIONALITY_MAP = {
    "CS": "regional",
    "CC": "regional_implicit",
    "CA": "region_agnostic",
    "-": "region_agnostic",
    "": "region_agnostic",
}

RANDOM_SEED = 42

VERY_HIGH_PRIORITY_SUBJECTS = {
    "professional_law",
    "international_law",
    "jurisprudence",
    "high_school_government_and_politics",
    "high_school_geography",
    "high_school_world_history",
    "high_school_us_history",
    "high_school_european_history",
    "prehistory",
    "world_religions",
    "sociology",
    "security_studies",
    "us_foreign_policy",
}

HIGH_PRIORITY_SUBJECTS = {
    "philosophy",
    "moral_scenarios",
    "moral_disputes",
    "business_ethics",
    "public_relations",
    "management",
    "marketing",
}

MEDIUM_PRIORITY_SUBJECTS = {
    "miscellaneous",
    "professional_psychology",
    "high_school_psychology",
    "human_aging",
    "human_sexuality",
    "nutrition",
    "clinical_knowledge",
    "professional_medicine",
    "college_medicine",
    "medical_genetics",
    "virology",
    "professional_accounting",
    "high_school_macroeconomics",
    "high_school_microeconomics",
    "econometrics",
    "logical_fallacies",
    "global_facts",
}

LOW_PRIORITY_SUBJECTS = {
    "elementary_mathematics",
    "high_school_mathematics",
    "high_school_statistics",
    "college_mathematics",
    "abstract_algebra",
    "formal_logic",
    "conceptual_physics",
    "high_school_physics",
    "college_physics",
    "high_school_chemistry",
    "college_chemistry",
    "high_school_biology",
    "college_biology",
    "anatomy",
    "astronomy",
    "electrical_engineering",
    "machine_learning",
    "computer_security",
    "college_computer_science",
    "high_school_computer_science",
}

def subject_keep_weight(subject):
    if subject in VERY_HIGH_PRIORITY_SUBJECTS:
        return 0.85
    if subject in HIGH_PRIORITY_SUBJECTS:
        return 0.65
    if subject in MEDIUM_PRIORITY_SUBJECTS:
        return 0.35
    if subject in LOW_PRIORITY_SUBJECTS:
        return 0.15
    return 0.25

# Clean records and sample

In [11]:
def clean_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip()


def make_prompt(row):
    return (
        f"{clean_text(row['question'])}\n\n"
        f"A) {clean_text(row['A'])}\n"
        f"B) {clean_text(row['B'])}\n"
        f"C) {clean_text(row['C'])}\n"
        f"D) {clean_text(row['D'])}"
    )


def safe_id_part(x):
    return clean_text(x).replace(" ", "_").replace("/", "_").replace("\\", "_")


def write_jsonl(records, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")


def read_jsonl(path):
    loaded = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            loaded.append(json.loads(line))
    return loaded

In [45]:
df = df_raw.copy()

df["question"] = df["question"].apply(clean_text)
df["option_a"] = df["option_a"].apply(clean_text)
df["option_b"] = df["option_b"].apply(clean_text)
df["option_c"] = df["option_c"].apply(clean_text)
df["option_d"] = df["option_d"].apply(clean_text)

df["answer"] = df["answer"].apply(clean_text).str.upper()
df["cultural_sensitivity_label"] = df["cultural_sensitivity_label"].apply(clean_text)
df["regionality"] = df["cultural_sensitivity_label"].map(REGIONALITY_MAP)

unknown_labels = sorted(df.loc[df["regionality"].isna(), "cultural_sensitivity_label"].dropna().unique())
print("Unknown cultural_sensitivity_label values:", unknown_labels)
assert df["regionality"].notna().all(), "Some cultural_sensitivity_label values were not mapped."

n_before = len(df)

df = df[
    df["question"].ne("")
    & df["option_a"].ne("")
    & df["option_b"].ne("")
    & df["option_c"].ne("")
    & df["option_d"].ne("")
].copy()

df = df[df["answer"].isin(VALID_ANSWERS)].copy()

df = df.rename(
    columns={
        "option_a": "A",
        "option_b": "B",
        "option_c": "C",
        "option_d": "D",
    }
)

dedup_cols = ["language", "question", "A", "B", "C", "D"]
n_before_dedup = len(df)
df = df.drop_duplicates(subset=dedup_cols).reset_index(drop=True)
n_after_dedup = len(df)

print(f"Rows before cleaning:       {n_before}")
print(f"Rows before deduplication:  {n_before_dedup}")
print(f"Rows after deduplication:   {n_after_dedup}")
print(f"Removed total:              {n_before - n_after_dedup}")

df_clean = df.copy()

# subject-weight sampling, but preserve both kept and discarded parts
df = df_clean.copy()
df["subject_keep_weight"] = df["subject"].apply(subject_keep_weight)
df.loc[df["regionality"] == "regional", "subject_keep_weight"] = 1.0

rng = np.random.default_rng(RANDOM_SEED)
df["sampling_random_value"] = rng.random(len(df))
df["keep_by_subject_weight"] = df["sampling_random_value"] <= df["subject_keep_weight"]

df_kept = df[df["keep_by_subject_weight"]].copy().reset_index(drop=True)
df_discarded = df[~df["keep_by_subject_weight"]].copy().reset_index(drop=True)

print(f"Rows before subject sampling: {len(df)}")
print(f"Rows kept in original sampled set: {len(df_kept)}")
print(f"Rows discarded by sampling: {len(df_discarded)}")

print("\nKept regionality counts:")
display(df_kept["regionality"].value_counts().sort_index().to_frame("n_examples"))

print("\nDiscarded regionality counts:")
display(df_discarded["regionality"].value_counts().sort_index().to_frame("n_examples"))

print("\nDiscarded language x regionality:")
display(
    df_discarded.groupby(["language", "regionality"])
    .size()
    .reset_index(name="n_examples")
    .pivot(index="language", columns="regionality", values="n_examples")
    .fillna(0)
    .astype(int)
)

print("\nDiscarded subject counts, top 50:")
display(df_discarded["subject"].value_counts().head(50).to_frame("n_examples"))

Unknown cultural_sensitivity_label values: []
Rows before cleaning:       70210
Rows before deduplication:  70198
Rows after deduplication:   69840
Removed total:              370
Rows before subject sampling: 69840
Rows kept in original sampled set: 35146
Rows discarded by sampling: 34694

Kept regionality counts:


,n_examples
regionality,
region_agnostic,31186
regional,3960



Discarded regionality counts:


,n_examples
regionality,
region_agnostic,34694



Discarded language x regionality:


regionality,region_agnostic
language,
chinese,6986
hindi,6935
italian,6951
russian,6914
spanish,6908



Discarded subject counts, top 50:


,n_examples
subject,
miscellaneous,2494
professional_psychology,1936
high_school_psychology,1685
elementary_mathematics,1565
moral_scenarios,1491
high_school_biology,1337
high_school_macroeconomics,1253
high_school_mathematics,1125
professional_law,1099


In [53]:
# summarize sampled and recovered parts
df_discarded_nonregional = df_discarded[df_discarded["regionality"] != "regional"].copy().reset_index(drop=True)

print("Sampled kept rows:", len(df_kept))
print("Discarded non-regional rows to add back:", len(df_discarded_nonregional))

print("\nKept language counts")
display(df_kept["language"].value_counts().sort_index().to_frame("n_examples"))

print("\nKept regionality counts")
display(df_kept["regionality"].value_counts().sort_index().to_frame("n_examples"))

print("\nDiscarded non-regional language x regionality")
display(
    df_discarded_nonregional.groupby(["language", "regionality"])
    .size()
    .reset_index(name="n_examples")
    .pivot(index="language", columns="regionality", values="n_examples")
    .fillna(0)
    .astype(int)
)

print("\nDiscarded non-regional subject counts, top 50")
display(df_discarded_nonregional["subject"].value_counts().head(50).to_frame("n_examples"))

Sampled kept rows: 35146
Discarded non-regional rows to add back: 34694

Kept language counts


,n_examples
language,
chinese,6978
hindi,7056
italian,6984
russian,7060
spanish,7068



Kept regionality counts


,n_examples
regionality,
region_agnostic,31186
regional,3960



Discarded non-regional language x regionality


regionality,region_agnostic
language,
chinese,6986
hindi,6935
italian,6951
russian,6914
spanish,6908



Discarded non-regional subject counts, top 50


,n_examples
subject,
miscellaneous,2494
professional_psychology,1936
high_school_psychology,1685
elementary_mathematics,1565
moral_scenarios,1491
high_school_biology,1337
high_school_macroeconomics,1253
high_school_mathematics,1125
professional_law,1099


# Build records

In [55]:
def make_record(row, idx):
    sample_id = safe_id_part(row.get("sample_id", ""))
    if sample_id == "":
        sample_id = f"{row['source_split']}_{int(row['source_idx'])}"

    return {
        "id": f"global_mmlu_{row['language']}_{sample_id}",
        "dataset": "global_mmlu",
        "language": row["language"],
        "split": "all",
        "task_type": "mcq",
        "prompt": make_prompt(row),
        "answer": row["answer"],
        "metadata": {
            "subject": clean_text(row.get("subject", "")),
            "subject_category": clean_text(row.get("subject_category", "")),
            "cultural_sensitivity_label": clean_text(row.get("cultural_sensitivity_label", "")),
            "regionality": row["regionality"],
            "region": clean_text(row.get("region", "")),
            "country": clean_text(row.get("country", "")),
            "sample_id": clean_text(row.get("sample_id", "")),
            "source_split": row["source_split"],
            "source_idx": int(row["source_idx"]),
            "subject_keep_weight": float(row["subject_keep_weight"]),
        },
    }


# make records
kept_records = [make_record(row, idx) for idx, row in df_kept.iterrows()]
discarded_nonregional_records = [make_record(row, idx) for idx, row in df_discarded_nonregional.iterrows()]
records_by_id = {}

for record in kept_records:
    records_by_id[record["id"]] = record

for record in discarded_nonregional_records:
    if record["id"] in records_by_id:
        raise ValueError(f"Duplicate recovered id: {record['id']}")
    records_by_id[record["id"]] = record

records = list(records_by_id.values())
print(f"Built kept records: {len(kept_records)}")
print(f"Built discarded non-regional records: {len(discarded_nonregional_records)}")
print(f"Built merged all records: {len(records)}")
print(json.dumps(records[0], ensure_ascii=False, indent=2))

Built kept records: 35146
Built discarded non-regional records: 34694
Built merged all records: 69840
{
  "id": "global_mmlu_russian_abstract_algebra_test_4",
  "dataset": "global_mmlu",
  "language": "russian",
  "split": "all",
  "task_type": "mcq",
  "prompt": "Найдите произведение данных многочленов в данном кольце многочленов. f(x) = 4x - 5, g(x) = 2x^2 - 4x + 2 в Z_8[x].\n\nA) 2x^2 + 5\nB) 6x^2 + 4x + 6\nC) 0\nD) x^2 + 1",
  "answer": "B",
  "metadata": {
    "subject": "abstract_algebra",
    "subject_category": "STEM",
    "cultural_sensitivity_label": "-",
    "regionality": "region_agnostic",
    "region": "[]",
    "country": "[]",
    "sample_id": "abstract_algebra/test/4",
    "source_split": "test",
    "source_idx": 4,
    "subject_keep_weight": 0.15
  }
}


In [56]:
def validate_record(record):
    assert record["dataset"] == "global_mmlu"
    assert record["language"] in LANGUAGE_CONFIGS
    assert record["split"] == "all"
    assert record["task_type"] == "mcq"
    assert record["answer"] in VALID_ANSWERS
    assert record["metadata"]["regionality"] in VALID_REGIONALITIES
    assert "\n\nA) " in record["prompt"]
    assert "\nB) " in record["prompt"]
    assert "\nC) " in record["prompt"]
    assert "\nD) " in record["prompt"]
    assert record["prompt"].strip()
    assert isinstance(record["metadata"]["source_idx"], int)


for record in records:
    validate_record(record)

print("All in-memory Global MMLU records are valid.")

All in-memory Global MMLU records are valid.


# Save and validate

In [58]:
records_by_language = {}

for language in LANGUAGE_CONFIGS:
    language_records = [
        record for record in records
        if record["language"] == language
    ]
    records_by_language[language] = language_records
    regional_records = [
        record for record in language_records
        if record["metadata"]["regionality"] == "regional"
    ]
    regional_path = OUTPUT_DIR / f"global_mmlu_{language}_regional.jsonl"
    write_jsonl(regional_records, regional_path)
    print(f"Saved {len(regional_records)} regional records to:")
    print(regional_path)

Saved 792 regional records to:
/scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/processed/global_mmlu/global_mmlu_russian_regional.jsonl
Saved 792 regional records to:
/scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/processed/global_mmlu/global_mmlu_chinese_regional.jsonl
Saved 792 regional records to:
/scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/processed/global_mmlu/global_mmlu_italian_regional.jsonl
Saved 792 regional records to:
/scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/processed/global_mmlu/global_mmlu_spanish_regional.jsonl
Saved 792 regional records to:
/scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/processed/global_mmlu/global_mmlu_hindi_regional.jsonl


In [59]:
# save only merged all files; do not overwrite regional-specific files
records_by_language = {}
for language in LANGUAGE_CONFIGS:
    language_records = [
        record for record in records
        if record["language"] == language
    ]

    ids = [record["id"] for record in language_records]
    assert len(ids) == len(set(ids)), f"Duplicate ids for {language}"

    records_by_language[language] = language_records

    all_path = OUTPUT_DIR / f"global_mmlu_{language}_all.jsonl"
    write_jsonl(language_records, all_path)

    print(f"Saved {len(language_records)} merged all records to:")
    print(all_path)

Saved 13974 merged all records to:
/scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/processed/global_mmlu/global_mmlu_russian_all.jsonl
Saved 13964 merged all records to:
/scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/processed/global_mmlu/global_mmlu_chinese_all.jsonl
Saved 13935 merged all records to:
/scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/processed/global_mmlu/global_mmlu_italian_all.jsonl
Saved 13976 merged all records to:
/scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/processed/global_mmlu/global_mmlu_spanish_all.jsonl
Saved 13991 merged all records to:
/scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/processed/global_mmlu/global_mmlu_hindi_all.jsonl


In [60]:
saved_paths = sorted(OUTPUT_DIR.glob("global_mmlu_*.jsonl"))
print(f"Saved JSONL files: {len(saved_paths)}")

for path in saved_paths:
    loaded = read_jsonl(path)
    print(f"{path.name}: {len(loaded)}")

    seen_ids = set()
    seen_prompts = set()

    for record in loaded:
        validate_record(record)

        assert record["id"] not in seen_ids
        seen_ids.add(record["id"])

        prompt_key = (record["language"], record["prompt"])
        assert prompt_key not in seen_prompts
        seen_prompts.add(prompt_key)

    if path.name.endswith("_regional.jsonl"):
        assert all(record["metadata"]["regionality"] == "regional" for record in loaded)

print("All saved Global MMLU files are valid.")

Saved JSONL files: 15
global_mmlu_chinese_all.jsonl: 13964
global_mmlu_chinese_region_agnostic.jsonl: 6186
global_mmlu_chinese_regional.jsonl: 792
global_mmlu_hindi_all.jsonl: 13991
global_mmlu_hindi_region_agnostic.jsonl: 6264
global_mmlu_hindi_regional.jsonl: 792
global_mmlu_italian_all.jsonl: 13935
global_mmlu_italian_region_agnostic.jsonl: 6192
global_mmlu_italian_regional.jsonl: 792
global_mmlu_russian_all.jsonl: 13974
global_mmlu_russian_region_agnostic.jsonl: 6268
global_mmlu_russian_regional.jsonl: 792
global_mmlu_spanish_all.jsonl: 13976
global_mmlu_spanish_region_agnostic.jsonl: 6276
global_mmlu_spanish_regional.jsonl: 792
All saved Global MMLU files are valid.


In [72]:
# save non-regional records from current all files
LANGUAGE_SHUFFLE_SEEDS = {
    "russian": 42,
    "chinese": 43,
    "italian": 44,
    "spanish": 45,
    "hindi": 46,
}
for language in LANGUAGE_CONFIGS:
    all_path = OUTPUT_DIR / f"global_mmlu_{language}_all.jsonl"
    nonregional_path = OUTPUT_DIR / f"global_mmlu_{language}_shuffled_region_agnostic.jsonl"
    all_records = read_jsonl(all_path)
    nonregional_records = [
        record for record in all_records
        if record["metadata"]["regionality"] != "regional"
    ]
    ids = [record["id"] for record in nonregional_records]
    assert len(ids) == len(set(ids)), f"duplicate ids in {nonregional_path}"
    rng = random.Random(LANGUAGE_SHUFFLE_SEEDS[language])
    rng.shuffle(nonregional_records)
    write_jsonl(nonregional_records, nonregional_path)
    print(language)
    print(f"  all records:          {len(all_records)}")
    print(f"  non-regional records: {len(nonregional_records)}")
    print(f"  shuffle seed:         {LANGUAGE_SHUFFLE_SEEDS[language]}")
    print(f"  saved to:             {nonregional_path}")

russian
  all records:          13974
  non-regional records: 13182
  shuffle seed:         42
  saved to:             /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/processed/global_mmlu/global_mmlu_russian_shuffled_region_agnostic.jsonl
chinese
  all records:          13964
  non-regional records: 13172
  shuffle seed:         43
  saved to:             /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/processed/global_mmlu/global_mmlu_chinese_shuffled_region_agnostic.jsonl
italian
  all records:          13935
  non-regional records: 13143
  shuffle seed:         44
  saved to:             /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/processed/global_mmlu/global_mmlu_italian_shuffled_region_agnostic.jsonl
spanish
  all records:          13976
  non-regional records: 13184
  shuffle seed:         45
  saved to:             /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingua

In [61]:
loaded_all_records = []

for language in LANGUAGE_CONFIGS:
    path = OUTPUT_DIR / f"global_mmlu_{language}_all.jsonl"
    loaded_all_records.extend(read_jsonl(path))

summary_df = pd.DataFrame(
    {
        "id": record["id"],
        "language": record["language"],
        "answer": record["answer"],
        "subject": record["metadata"]["subject"],
        "subject_category": record["metadata"]["subject_category"],
        "cultural_sensitivity_label": record["metadata"]["cultural_sensitivity_label"],
        "regionality": record["metadata"]["regionality"],
        "region": record["metadata"]["region"],
        "country": record["metadata"]["country"],
        "source_split": record["metadata"]["source_split"],
        "subject_keep_weight": record["metadata"]["subject_keep_weight"],
    }
    for record in loaded_all_records
)

print("Final summary")
print("-------------")
print(f"Raw loaded rows:       {len(df_raw)}")
print(f"After sampling:        {len(df)}")
print(f"Saved all records:     {len(loaded_all_records)}")
print(f"Output dir:            {OUTPUT_DIR}")

print("\nBy language:")
display(summary_df["language"].value_counts().sort_index().to_frame("n_examples"))

print("\nBy regionality:")
display(summary_df["regionality"].value_counts().sort_index().to_frame("n_examples"))

print("\nBy language and regionality:")
display(
    summary_df.groupby(["language", "regionality"])
    .size()
    .reset_index(name="n_examples")
    .pivot(index="language", columns="regionality", values="n_examples")
    .fillna(0)
    .astype(int)
)

print("\nBy subject_category:")
display(summary_df["subject_category"].value_counts(dropna=False).sort_index().to_frame("n_examples"))

print("\nBy subject, top 50:")
display(summary_df["subject"].value_counts().head(50).to_frame("n_examples"))

Final summary
-------------
Raw loaded rows:       70210
After sampling:        69840
Saved all records:     69840
Output dir:            /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/processed/global_mmlu

By language:


,n_examples
language,
chinese,13964
hindi,13991
italian,13935
russian,13974
spanish,13976



By regionality:


,n_examples
regionality,
region_agnostic,65880
regional,3960



By language and regionality:


regionality,region_agnostic,regional
language,,
chinese,13172,792
hindi,13199,792
italian,13143,792
russian,13182,792
spanish,13184,792



By subject_category:


,n_examples
subject_category,
Business,3595
Humanities,23523
Medical,7270
Other,4413
STEM,15717
Social Sciences,15322



By subject, top 50:


,n_examples
subject,
professional_law,7670
moral_scenarios,4475
miscellaneous,3913
professional_psychology,3051
high_school_psychology,2681
high_school_macroeconomics,1950
elementary_mathematics,1884
moral_disputes,1730
prehistory,1620


In [ ]:
# subjects saved as regional
regional_subjects = (
    df[df["regionality"] == "regional"]
    .groupby(["language", "subject_category", "subject"])
    .size()
    .reset_index(name="n_examples")
    .sort_values(["language", "n_examples"], ascending=[True, False])
)

# non regional
regional_subjects = set(regional_subjects['subject'].tolist())
non_regional_subjects = set(summary_df['subject'].tolist()) - regional_subjects
non_regional_subjects

{'abstract_algebra',
 'anatomy',
 'clinical_knowledge',
 'college_computer_science',
 'college_physics',
 'computer_security',
 'conceptual_physics',
 'econometrics',
 'formal_logic',
 'high_school_biology',
 'machine_learning',
 'medical_genetics'}